# Experiment 12 — Demographic subgroup analysis

Disease classification AUC (exp01) and downstream $\Delta$AUC (exp04) are
stratified by sex (M/F) and age tertile, per dataset. NIH demographics come
from `Data_Entry_2017_v2020.csv` (`Patient Gender`, `Patient Age`); MIMIC
demographics come from `patients.csv.gz` joined on `subject_id`. Emory
demographics are produced on the PHI-compatible infrastructure and reported
qualitatively in the main manuscript.


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [ ]:
import os, warnings
from pathlib import Path
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators_work'))

def _nih_demographics():
    f = Path('/data0/NIH-CXR14/Data_Entry_2017_v2020.csv')
    if not f.exists(): return None
    d = pd.read_csv(f)
    return (d.rename(columns={'Patient Gender': 'sex', 'Patient Age': 'age',
                              'Image Index': 'image'})
              [['image', 'sex', 'age']])

def _mimic_demographics():
    f = Path('/data0/MIMIC-CXR/patients.csv.gz')
    if not f.exists(): return None
    d = pd.read_csv(f)[['subject_id', 'gender', 'anchor_age']]
    return d.rename(columns={'gender': 'sex', 'anchor_age': 'age'})

out_dir = ROOT / 'v4_exp12_demographic_subgroups'
out_dir.mkdir(parents=True, exist_ok=True)

records = []
for ds, get_demo in [('nih', _nih_demographics), ('mimic', _mimic_demographics)]:
    demo = get_demo()
    if demo is None:
        warnings.warn(f'{ds}: demographics source absent; subgroup rows skipped.')
        continue
    preds_glob = sorted(ROOT.glob(f'v4_exp01_disease_classification_{ds}/preds_*.parquet'))
    if not preds_glob:
        warnings.warn(f'{ds}: exp01 preds cache absent; subgroup rows skipped.')
        continue
    for pf in preds_glob:
        preds = pd.read_parquet(pf)
        key = 'image_path' if ds == 'nih' else 'subject_id'
        right_on = 'image' if ds == 'nih' else 'subject_id'
        merged = preds.merge(demo, how='left', left_on=key, right_on=right_on)
        if merged['sex'].isna().all(): continue
        merged['age_tertile'] = pd.qcut(merged['age'], 3,
            labels=['young', 'mid', 'old'], duplicates='drop')
        for grp_col in ('sex', 'age_tertile'):
            for grp, sub in merged.groupby(grp_col, observed=True):
                if sub['y'].nunique() < 2: continue
                records.append({'dataset': ds, 'source': pf.name,
                                'grouping': grp_col, 'group': str(grp),
                                'n': len(sub), 'n_pos': int(sub['y'].sum()),
                                'auc': roc_auc_score(sub['y'], sub['y_hat'])})

out = pd.DataFrame(records)
if out.empty:
    pd.DataFrame({'status': ['inputs_absent']}).to_parquet(
        out_dir / 'exp12_manifest.parquet', index=False)
else:
    out.to_parquet(out_dir / 'exp12_subgroup_auc.parquet', index=False)
    print(f'wrote {len(out)} rows -> {out_dir / "exp12_subgroup_auc.parquet"}')
